# Step 26: Is the best model really better?
We compare models **person by person**: every speaker (18) gets predictions from a model that never saw their conversation. Then a **Wilcoxon test**, a **bootstrap 95% interval** and **Holm correction**.
*Same as `src/26_significance.py`.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

In [ ]:
def past_mean(X, speakers, k):
    # for every row: the mean of the same speaker's last k seconds (only the past)
    out = np.zeros(X.shape)
    for s in pd.unique(speakers):
        rows = np.where(speakers == s)[0]
        out[rows] = pd.DataFrame(X[rows]).rolling(k, min_periods=1).mean().to_numpy()
    return out

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
bookkeeping = ["face_0", "face_1", "face_2", "face_3", "face_4"]
cols, voice = [], []
for c in df.columns:
    if (c.startswith("voice_") or c.startswith("face_")) and c not in bookkeeping:
        cols.append(c)
    if c.startswith("voice_"):
        voice.append(c)
y = df["engagement"].to_numpy("float32")
speaker = (df["session"] + "_" + df["role"]).to_numpy()
session = df["session"].to_numpy()

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**The three models:** A = baseline, B = + context, C = best (fusion + context).

In [ ]:
own = df[cols].fillna(0).to_numpy("float32")
fused = np.hstack([df[voice].fillna(0).to_numpy("float32"),
                   np.load("data/features/pretrained_w2vbert2.npy"),
                   np.load("data/features/pretrained_videomae.npy")])
inputs = {
    "A": own,
    "B": np.hstack([own, past_mean(own, speaker, 5).astype("float32"), past_mean(own, speaker, 10).astype("float32")]),
    "C": np.hstack([fused, past_mean(fused, speaker, 5).astype("float32"), past_mean(fused, speaker, 10).astype("float32")]),
}

**Predict every row once**, with a model that did not see its conversation.

In [ ]:
predictions = {}
for name, X in inputs.items():
    pred = np.zeros(len(y), dtype="float32")
    for test_sessions in folds:
        test = np.isin(session, test_sessions)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[~test])
        model = HistGradientBoostingRegressor(max_iter=300, random_state=42)
        model.fit(X_train, y[~test])
        pred[test] = model.predict(scaler.transform(X[test]))
    predictions[name] = pred
    print(name, "done")

**CCC for each of the 18 speakers**, and the average per conversation (9).

In [ ]:
per_speaker = pd.DataFrame(index=pd.unique(speaker))
for name in predictions:
    values = []
    for s in per_speaker.index:
        values.append(ccc(y[speaker == s], predictions[name][speaker == s]))
    per_speaker[name] = values

conversation = per_speaker.index.str.split("_").str[0]        # "001_expert" -> "001"
per_conversation = per_speaker.groupby(conversation).mean()
per_speaker.round(3)

**The tests.** For each pair and level: average difference, 95% bootstrap interval, how many improved, Wilcoxon p.

In [ ]:
rng = np.random.default_rng(42)
pairs = [("C", "A"), ("B", "A"), ("C", "B")]
results = []
for a, b in pairs:
    for level, table in [("speaker", per_speaker), ("conversation", per_conversation)]:
        diff = (table[a] - table[b]).to_numpy()
        boot = rng.choice(diff, (10000, len(diff))).mean(axis=1)    # 10,000 resamples
        results.append([a + " vs " + b, level, diff.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5),
                        (diff > 0).sum(), len(diff), wilcoxon(diff).pvalue])
res = pd.DataFrame(results, columns=["pair", "level", "difference", "ci_low", "ci_high", "better", "out_of", "p"])

**Holm correction** (3 tests per level): sort the p-values; multiply the smallest by 3, the next by 2, the last by 1 (never going down).

In [ ]:
res["p_holm"] = 0.0
for level in ["speaker", "conversation"]:
    rows = res[res["level"] == level].sort_values("p").index
    running = 0
    for rank, i in enumerate(rows):
        running = max(running, min(1, (3 - rank) * res.loc[i, "p"]))
        res.loc[i, "p_holm"] = running
res["significant"] = res["p_holm"] < 0.05
res.round(4)

**The result saved by the script**, for comparison.

In [ ]:
print(pd.read_csv("reports/significance_results.csv")[["comparison", "level", "mean_diff", "ci_low", "ci_high", "p_holm"]].round(4))